In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
# import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dino_model import DINOFeaturePresence #, DINOWithMIL
# from nocode_robot_programming.state_decision.state_decider_alt import DINOProtoPresence
# # from nocode_robot_programming.state_decision.dino_model_v2 import DINOFeaturePresence
# # from nocode_robot_programming.state_decision.dino_model_v3 import DINOFeaturePresence
# from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
# from nocode_robot_programming.state_decision.AEGP_model import AEGP
# from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
# from nocode_robot_programming.state_decision.utils import Filename
# from gesture_detector.utils import pretty_confusion_matrix
# from trajectory_data.skill_visualizer import play_video
# from nocode_robot_programming.state_decision.utils import add_tag
# from nocode_robot_programming.state_decision.utils import visualize_accuracies
# from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing
# from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

import trajectory_data
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dataset_D1 import d1_peg_pick #, d2_peg_pick, d1_probe, d2_probe, dupl, get_dataset_view

import torch
import torch.nn.functional as F

import numpy as np
seed = 49
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

datafileloader = TrajectoryDataset(trajectory_data.package_path)
datasets = d1_peg_pick(datafileloader)

No other ipykernel_launcher processes found.
Found tasks:
Task        | # Files | Names (unique)                                                  | Trials | Offsets
------------+---------+-----------------------------------------------------------------+--------+--------
d1_peg_pick | 11      | d1_peg_pick, d1_peg_pick_branch_at_49, … (+9 more)              | -1-8   | 0-49   
d1_probe    | 9       | d1_probe, d1_probe_branch_at_51, d1_probe_trial_0, … (+6 more)  | -1-6   | 0-51   
d2_peg_pick | 7       | d2_peg_pick, d2_peg_pick_branch_at_76, … (+5 more)              | -1-4   | 0-76   
d2_probe    | 9       | d2_probe, d2_probe_branch_at_103, d2_probe_trial_0, … (+6 more) | -1-6   | 0-103  
d3_peg_pick | 5       | d3_peg_pick, d3_peg_pick_branch_at_189, … (+3 more)             | -1-2   | 0-189  
d3_probe    | 5       | d3_probe, d3_probe_branch_at_118, d3_probe_trial_0, … (+2 more) | -1-2   | 0-118  


# First DINO model `DINOFeaturePresence`

DINO is pretrained model that can give an input image a embeddings - some features that something is in the image.

In [2]:
X = datasets[0][0].X
X.shape

torch.Size([21, 64, 64])

In [3]:
dino_model = torch.hub.load('facebookresearch/dinov2', "dinov2_vits14").to("cuda").eval()

Using cache found in /home/steve/.cache/torch/hub/facebookresearch_dinov2_main


In [4]:
X = X.to("cuda", non_blocking=True)
X = X.unsqueeze(1).repeat(1, 3, 1, 1)
X = torch.nn.functional.interpolate(
    X, size=(224, 224),
    mode="bilinear", align_corners=False
)
mean = torch.tensor([0.485, 0.456, 0.406], device="cuda")[None, :, None, None]
std  = torch.tensor([0.229, 0.224, 0.225], device="cuda")[None, :, None, None]
X = (X - mean) / std
X.shape

torch.Size([21, 3, 224, 224])

In [5]:
out = dino_model.forward_features(X)
P = out["x_norm_patchtokens"].float()  # [B, M, D]

In [6]:
out.keys()

dict_keys(['x_norm_clstoken', 'x_norm_regtokens', 'x_norm_patchtokens', 'x_prenorm', 'masks'])

In [7]:
P.shape

torch.Size([21, 256, 384])

These are outputs from the DINO model `x_norm_patchtokens`:

We can see 21 samples, 256 patches - 16x16 patches per image so for every image the model gets these embeddings or features for each patch, we see 384 dimension of features. So, anything that is in each patch is expressed as 384 numbers.

`x_norm_clstoken` is the feature vector for the whole image:

In [8]:
P = out["x_norm_clstoken"].float()
P.shape

torch.Size([21, 384])

In [9]:
P = out["x_norm_regtokens"].float()
P.shape

torch.Size([21, 0, 384])

In [10]:
P = out["x_prenorm"].float()
P.shape

torch.Size([21, 257, 384])

In [11]:
P = out["masks"]
P is None

True

DINO-based classifier with cosine prototypes over patch tokens.

- Trains by computing per-class prototypes from training images.
- Predicts by pooling image patch features, then cosine to class prototypes.
- Optional open-set gating via per-class percentile threshold on train scores.

In [39]:
self = DINOFeaturePresence()
C = 2
X = datasets[0][0].X
y = datasets[0][0].y_int
y_cls = datasets[0][0].y_cls

Using cache found in /home/steve/.cache/torch/hub/facebookresearch_dinov2_main


## Patch features for all images

In [ ]:
X = X.to(self.device, non_blocking=True)
feats = []
for i in range(0, X.size(0), self.batch_size):
    xb = self._prep_batch(X[i:i+self.batch_size])         # [B,3,S,S]
    out = self.model.forward_features(xb)
    P = out["x_norm_patchtokens"].float()                 # [B, M, D]
    feats.append(F.normalize(P, dim=-1))
P = torch.cat(feats, dim=0)
N, M, D = P.shape

In [43]:
N, M, D

(21, 256, 384)

Ok, we have a dataset 21 samples, 256 patches and 384 feature dimension - as before

## Pool image -> single vector (default: mean over patches)

In [ ]:
G = P.mean(dim=1)                      # [N, D]
G = F.normalize(G, dim=-1)

In [45]:
G.shape

torch.Size([21, 384])

## Build class prototypes as mean of pooled features, then L2-normalize

In [ ]:
prototypes = torch.zeros(C, D, device=self.device)
for c in range(C): # For every class
    mask = (y == c)
    assert mask.any(), f"No samples for class id {c}"
    mu = G[mask].mean(dim=0)
    prototypes[c] = F.normalize(mu, dim=0)
self.prototypes = prototypes                   # [C, D]

## Per-class open-set thresholds from training scores

In [ ]:
if self.percentile_keep is not None:
    with torch.no_grad():
        logits = self._score_logits(G, self.prototypes)  # [N, C]
        pos_scores = []                                  # list of tensors (per class)
        for c in range(C):
            pos_scores.append(logits[y == c, c].detach().float().cpu())
    thresholds = []
    for c in range(C):
        s = pos_scores[c].sort().values
        k = max(0, min(len(s)-1, int(round(self.percentile_keep * (len(s)-1)))))
        thresholds.append(s[k].item())
    self.thresholds = torch.tensor(thresholds, device=self.device)

What I don't like about this: 
- Mean across all patches

# Predict

In [ ]:
sample = X[0]

# p = dino_model._single_patch_feats(X[0])  # [M, D]
xb = self._prep_batch(sample.unsqueeze(0))
out = self.model.forward_features(xb)
P = out["x_norm_patchtokens"].float()[0]  # [M, D]
p = F.normalize(P, dim=-1)

# g = self._pool_patches(p.unsqueeze(0))[0]  # [D]
G = P.unsqueeze(0).mean(dim=1)  # [N, D]
g = F.normalize(G, dim=-1)

# logits = self._score_logits(g.unsqueeze(0), self.prototypes)[0]  # [C]
logits = g @ self.prototypes.T
c = int(torch.argmax(logits).item())
y_cls[c]


'd1_peg_pick'

: 

This was the simplest method that uses DINO

- 224x224 input image better than 64x64 - more pixels, more details
- DINO smaller model works better than large, perhaps, the small can spot simpler features that we can utilize better

# MIL Extension

- Learns a small attention head to pool patches into an image embedding.
- Classifier is cosine to learnable class weights (W_cls).
- Falls back to mean pooling if head isn't initialized (e.g., before train()).